# Session 1: two gears trade speed for torque
**A. Real-world problem.** A bicycle, a clock, a hand drill and a car gearbox all use gears.
Why does a small gear driving a big gear turn the big one slowly but with more twisting
force? And why do two meshing gears turn in opposite directions?

**Plan.** (1) one mesh: speed, torque and direction, (2) a chain of gears with idlers,
(3) check that power (torque times speed) is the same going in and coming out, (4) draw a train.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: one mesh
The teeth of two meshing gears pass the contact point one after the other, so the number of
teeth that go past must be equal on both gears: `N_driver * w_driver = N_driven * w_driven`.
So the driven gear turns at `w_out = w_in * N_driver / N_driven` (w = speed in rpm).

For an ideal mesh (no friction) the power stays the same, so the torque goes the other way:
`T_out = T_in * N_driven / N_driver`. The two gears turn in opposite directions; we show that
with a minus sign. Check by hand: 20 teeth driving 60 teeth at 120 rpm and 5 N m gives
w_out = 120 * 20 / 60 = 40 rpm and T_out = 5 * 60 / 20 = 15 N m, both with a minus sign.

In [ ]:
def mesh(w_in, torque_in, n_driver, n_driven):
    """One ideal external mesh. Returns (w_out, torque_out); the minus sign means 'turns the other way'."""
    ratio = n_driven / n_driver
    return -w_in / ratio, -torque_in * ratio

## Step 2: a chain of gears
Gears 1, 2, ..., n in a row, each meshing with the next. A gear in the middle is called an
*idler*: it is driven by one gear and drives the next. Every mesh flips the direction, so
after k meshes the direction is (-1)^k. The idler's tooth count cancels out of the ratio.

In [ ]:
def train(w_in, torque_in, teeth):
    """Speed and torque out of a row of gears with the given tooth counts (ideal)."""
    w, t = w_in, torque_in
    for n_a, n_b in zip(teeth[:-1], teeth[1:]):
        w, t = mesh(w, t, n_a, n_b)
    return w, t


def direction(k_meshes):
    """+1 if the last gear turns the same way as the first, -1 if the opposite way."""
    return (-1) ** k_meshes


def power_watts(torque_nm, w_rpm):
    """Power = torque x angular speed, with w converted from rpm to rad/s."""
    return torque_nm * w_rpm * 2 * np.pi / 60

## Step 3: draw a train
The picture needs a gear outline: a circle of radius `module * N / 2` (the pitch circle) with
N teeth around it. The module is the tooth size in mm; two gears can only mesh if they share it.

In [ ]:
def gear_outline(n_teeth, module, angle=0.0):
    """Closed polygon (x, y) of a simple gear with n_teeth teeth, turned by angle (radians)."""
    r = module * n_teeth / 2
    pitch = 2 * np.pi / n_teeth
    a = []
    rad = []
    for k in range(n_teeth):
        c = k * pitch
        a += [c - 0.40 * pitch, c - 0.18 * pitch, c + 0.18 * pitch, c + 0.40 * pitch]
        rad += [r - 1.25 * module, r + module, r + module, r - 1.25 * module]
    a = np.array(a) + angle
    rad = np.array(rad)
    return rad * np.cos(a), rad * np.sin(a)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m = 1.0
    teeth = [20, 30, 60]
    print("train", teeth, "->", train(120, 5, teeth))
    x = 0.0
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ang = 0.0
    centres = []
    for i, n in enumerate(teeth):
        r = m * n / 2
        if i > 0:
            x += m * teeth[i - 1] / 2 + r
            ang = np.pi - (np.pi + ang * teeth[i - 1]) / n   # tooth meets gap at the contact point
        px, py = gear_outline(n, m, ang)
        ax.fill(px + x, py, alpha=0.5, label=f"gear {i + 1}: {n} teeth")
        centres.append(x)
        ax.text(x, 0, f"{n}", ha="center", va="center")
    ax.set_aspect("equal")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title("Gear 1 drives gear 3 through an idler: ratio 3, same direction")
    fig.tight_layout()
    fig.savefig(OUT / "gear_train.png", dpi=120)

    n_driven = np.arange(10, 61)
    w_out = 120 * 20 / n_driven
    t_out = 5 * n_driven / 20
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(n_driven, w_out, label="output speed (rpm)")
    ax2.set_xlabel("teeth on the driven gear (20-tooth driver)")
    ax2.set_ylabel("output speed (rpm)")
    ax3 = ax2.twinx()
    ax3.plot(n_driven, t_out, color="tab:red", label="output torque (N m)")
    ax3.set_ylabel("output torque (N m)")
    ax2.set_title("A bigger driven gear: slower and stronger")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "speed_torque_vs_teeth.png", dpi=120)
    w, t = train(120, 5, [20, 60])
    print("mesh 20 -> 60:", w, t, "power in", round(power_watts(5, 120), 2), "out", round(power_watts(t, w), 2))
    print("saved figures in", OUT)

## Try it
Change the tooth counts, predict the output speed and direction first, then run it. Answer the
four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` change the idler in `train(120, 5, [20, 30, 60])` from 30 to 45 teeth. Predict the output speed first, then run it.
2. **Stretch.** Write `direction_after(teeth)` that returns +1 or -1 for a row of gears (use the number of meshes). Test it on [20, 30, 25, 60] and say how many idlers it has.
3. **Explain.** Explain, in two sentences, why an idler does not change the speed ratio but does change the direction.

## Assessment
- **A. Conceptual.** If the driven gear has twice as many teeth as the driving gear, what is the ideal output speed ratio, and what real-world losses should be considered?
- **B. Numerical.** A 40-tooth gear drives a 20-tooth gear at 100 rpm and 4 N m. Find the output speed and ideal torque, and say what kind of train this is.
- **C. Graph and simulation interpretation.** In the speed and torque graphs for a 20-tooth driver at 120 rpm and 5 N m, read the output speed and torque when the driven gear has 40 teeth and when it has 60 teeth. What happens to their product?
- **D. Design and optimisation challenge.** Design: from a 20-tooth driver at 120 rpm and 5 N m you need 40 rpm out turning the same way as the input. Choose the driven gear and the idlers, and state the output torque. What is the price of an idler?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    w, t = L.mesh(120, 5, 20, 60)
    checks.append(("worked example: 20 teeth driving 60 at 120 rpm and 5 N m gives -40 rpm and -15 N m",
                   abs(w + 40) < 1e-9 and abs(t + 15) < 1e-9))
    checks.append(("worked example: power in equals power out (62.83 W)",
                   abs(L.power_watts(5, 120) - 62.832) < 1e-3 and abs(L.power_watts(t, w) - L.power_watts(5, 120)) < 1e-9))
    w2, t2 = L.mesh(100, 4, 40, 20)
    checks.append(("hand calculation: 40 driving 20 (a speed-up) gives -200 rpm and -2 N m",
                   abs(w2 + 200) < 1e-9 and abs(t2 + 2) < 1e-9))
    checks.append(("a driven gear with twice the teeth runs at half the speed with twice the torque",
                   np.allclose(L.mesh(60, 3, 15, 30), (-30.0, -6.0))))
    w3, t3 = L.train(120, 5, [20, 30, 60])
    checks.append(("an idler (30 teeth) between 20 and 60 gives +40 rpm and +15 N m: same direction",
                   abs(w3 - 40) < 1e-9 and abs(t3 - 15) < 1e-9))
    w4, _ = L.train(120, 5, [20, 77, 60])
    checks.append(("the idler tooth count does not change the speed", abs(w4 - w3) < 1e-9))
    checks.append(("direction is (-1)^k for k meshes: 1 mesh -1, 2 meshes +1, 3 meshes -1",
                   [L.direction(k) for k in (1, 2, 3)] == [-1, 1, -1]))
    w5, t5 = L.train(120, 5, [20, 30, 25, 60])
    checks.append(("power is conserved through a four-gear train",
                   abs(abs(L.power_watts(t5, w5)) - L.power_watts(5, 120)) < 1e-9))
    w7, t7 = L.mesh(120, 5, 20, 40)
    checks.append(("graph reading: 20 teeth driving 40 at 120 rpm and 5 N m gives -60 rpm and -10 N m",
                   abs(w7 + 60) < 1e-9 and abs(t7 + 10) < 1e-9))
    checks.append(("[20, 30, 25, 60] has 3 meshes (two idlers): -40 rpm, -15 N m",
                   abs(w5 + 40) < 1e-9 and abs(t5 + 15) < 1e-9))
    checks.append(("a speed-up 40 to 20 at 100 rpm and 4 N m carries 41.9 W in and out",
                   abs(L.power_watts(4, 100) - 41.89) < 0.01 and abs(L.power_watts(2, 200) - L.power_watts(4, 100)) < 1e-9))
    w6, _ = L.mesh(120, 5, 40, 40)
    checks.append(("edge case: equal gears turn at the same speed, opposite way", abs(w6 + 120) < 1e-9))
    x, y = L.gear_outline(20, 1.0)
    r = np.hypot(x, y)
    checks.append(("gear outline: teeth reach pitch radius 10 + module 1 = 11 and roots 10 - 1.25 = 8.75",
                   abs(r.max() - 11.0) < 1e-9 and abs(r.min() - 8.75) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
